In [56]:
import pandas as pd
import geopandas as gpd
import joblib
import numpy as np
import warnings

In [57]:
def muat_data_dan_model(file_model, file_input_cuaca):
    """Memuat model dan data input cuaca."""
    print("[1/5] Memuat model dan data input cuaca...")
    try:
        model = joblib.load(file_model)
        df_cuaca = pd.read_csv(file_input_cuaca)
        df_cuaca.rename(columns={
            'rata_rata_humi': 'humidity',
            'Curah_Hujan_Rata': 'rainfall',
            'rata_rata_temp': 'temperature'
        }, inplace=True, errors='ignore')
        return model, df_cuaca
    except FileNotFoundError as e:
        print(f"[ERROR] Gagal memuat file: {e}")
        return None, None

In [58]:
def get_kategori(prob):
    if prob >= 0.75:
        return "sangat ideal"
    elif prob >= 0.65:
        return "mendekati ideal"
    else:
        return "cukup ideal"

In [ ]:
def lakukan_prediksi_awal(model, df_cuaca):
    """
    DIUBAH: Melakukan prediksi awal dan langsung memformat outputnya
    agar sesuai dengan struktur final yang diinginkan.
    """
    print("[2/5] Melakukan prediksi awal berdasarkan cuaca...")
    fitur_wajib = ['temperature', 'humidity', 'rainfall']
    if not all(fitur in df_cuaca.columns for fitur in fitur_wajib):
        print(f"[ERROR] Data input cuaca tidak memiliki semua kolom yang dibutuhkan: {fitur_wajib}")
        return None
        
    fitur_cuaca = df_cuaca[fitur_wajib]
    prediksi_proba = model.predict_proba(fitur_cuaca)
    kelas_model = model.classes_
    
    hasil_awal = []
    for probas in prediksi_proba:
        top_indices = probas.argsort()[-3:][::-1]
        top_crops = []
        for i in top_indices:
            prob = probas[i]
            top_crops.append({
                'nama': kelas_model[i],
                'skor': f"{prob * 100:.0f}%",
                'kategori': get_kategori(prob) 
            })
        hasil_awal.append(top_crops)
        
    df_cuaca['prediksi_awal'] = hasil_awal
    return df_cuaca

In [ ]:
def gabung_dengan_data_lahan(df_cuaca, file_peta_lahan):
    """Menggabungkan data cuaca dengan data peta penggunaan lahan."""
    print(f"[3/4] Memuat Peta Penggunaan Lahan dari file lokal '{file_peta_lahan}'...")
    try:
        gdf_lahan = gpd.read_file(file_peta_lahan)
    except Exception as e:
        print(f"[ERROR] Gagal membaca file GeoJSON: {e}")
        return None

    gdf_cuaca = gpd.GeoDataFrame(
        df_cuaca, 
        geometry=gpd.points_from_xy(df_cuaca.LON, df_cuaca.LAT),
        crs="EPSG:4326"
    )
    
    if gdf_cuaca.crs != gdf_lahan.crs:
        gdf_cuaca = gdf_cuaca.to_crs(gdf_lahan.crs)
    
    print("      Melakukan Spatial Join untuk identifikasi jenis lahan...")
    gdf_hasil = gpd.sjoin(gdf_cuaca, gdf_lahan, how="left", predicate='within')
    
    return gdf_hasil.rename(columns={'namobj': 'jenis_lahan'})

In [ ]:
def terapkan_aturan_logika_lanjutan(df_hasil):
    """
    Menerapkan aturan bisnis yang lebih canggih untuk mengoreksi prediksi 
    berdasarkan berbagai jenis penggunaan lahan.
    """
    print("[4/4] Menerapkan aturan logika bisnis yang disempurnakan...")
    rekomendasi_final = []
    
    prioritas_tegalan = ['Jagung', 'Ubi Jalar', 'Kacang Tanah', 'Singkong'] 
    prioritas_kebun = ['Pisang', 'Mangga', 'Pepaya', 'Kopi', 'Jeruk'] 

    for index, row in df_hasil.iterrows():
        prediksi_awal = row['prediksi_awal']
        jenis_lahan = row['ptnobjname'] 
        
        prediksi_koreksi = prediksi_awal.copy()

        if pd.notna(jenis_lahan) and 'Sawah' in jenis_lahan:
            rekomendasi_padi = {
                'nama': 'Padi',
                'skor': '100%',
                'kategori': 'sangat ideal',
                'catatan': 'Prioritas utama untuk lahan sawah'
            }

            prediksi_koreksi = [p for p in prediksi_koreksi if p['nama'] != 'Padi']
            prediksi_koreksi.insert(0, rekomendasi_padi)

        elif pd.notna(jenis_lahan) and ('Tegalan' in jenis_lahan or 'Ladang' in jenis_lahan):
            prediksi_koreksi = [p for p in prediksi_koreksi if p['nama'] != 'Padi']
            
            for prioritas in prioritas_tegalan:
                for i, p in enumerate(prediksi_koreksi):
                    if p['nama'] == prioritas:
                        item = prediksi_koreksi.pop(i)
                        item['catatan'] = 'Prioritas untuk Lahan Kering/Tegalan'
                        prediksi_koreksi.insert(0, item)
                        break 

        elif pd.notna(jenis_lahan) and 'Kebun' in jenis_lahan:
            for prioritas in prioritas_kebun:
                for i, p in enumerate(prediksi_koreksi):
                    if p['nama'] == prioritas:
                        item = prediksi_koreksi.pop(i)
                        item['catatan'] = 'Cocok untuk Kebun & Pekarangan'
                        prediksi_koreksi.insert(0, item)
                        break
        
        elif pd.notna(jenis_lahan) and any(keyword in jenis_lahan for keyword in ['Kampung', 'Permukiman', 'Industri', 'Perkantoran']):
             prediksi_koreksi = [{'nama': 'Tidak Ada', 'skor': 0.0, 'catatan': f'Lahan tidak cocok untuk pertanian ({jenis_lahan})'}]
        
        else:
            pass 
            
        rekomendasi_final.append(prediksi_koreksi[:3])
        
    df_hasil['rekomendasi_final'] = rekomendasi_final
    return df_hasil

In [ ]:
MODEL_FILE = 'home/noturminesv/projects/gis-ml/model/ModelClassifier.pkl'
CUACA_INPUT_FILE = 'home/noturminesv/projects/gis-ml/dataset/period3_psch_ht.csv'
PETA_LAHAN_FILE = 'home/noturminesv/projects/gis-ml/dataset/Peta_Penggunaan_Lahan_DIY.geojson'

model, df_cuaca = muat_data_dan_model(MODEL_FILE, CUACA_INPUT_FILE)

if model and df_cuaca is not None:
    df_prediksi_awal = lakukan_prediksi_awal(model, df_cuaca)
    
    if df_prediksi_awal is not None:
        df_hasil_join = gabung_dengan_data_lahan(df_prediksi_awal, PETA_LAHAN_FILE)
        
        if df_hasil_join is not None:
            df_final = terapkan_aturan_logika_lanjutan(df_hasil_join)
            
            print("\n--- HASIL AKHIR REKOMENDASI YANG DISEMPURNAKAN ---")
            print(df_final[['LAT', 'LON', 'jenis_lahan', 'prediksi_awal', 'rekomendasi_final']].head())
            
            kolom_dihapus = [col for col in ['geometry', 'index_right', 'objectid', 'ptnid', 'ptnsbjname', 'ig25k_penggunaan10k_ar_area', 'ptndate', 'ptnremarks', 'ruleid', 'fcode', 'metadata', 'remark', 'srs_id', 'namobj', 'jnlp', 'ptnobjname', 'jenis_lahan'] if col in df_final.columns]
            df_final.drop(columns=kolom_dihapus, inplace=True)
            
            df_final.head()

[1/5] Memuat model dan data input cuaca...
[2/5] Melakukan prediksi awal berdasarkan cuaca...
[3/4] Memuat Peta Penggunaan Lahan dari file lokal '../../dataset/Peta_Penggunaan_Lahan_DIY.geojson'...
      Melakukan Spatial Join untuk identifikasi jenis lahan...
[4/4] Menerapkan aturan logika bisnis yang disempurnakan...

--- HASIL AKHIR REKOMENDASI YANG DISEMPURNAKAN ---
    LAT     LON     jenis_lahan  \
0 -8.15  110.65  Tegalan/Ladang   
1 -8.15  110.70  Tegalan/Ladang   
2 -8.15  110.75  Tegalan/Ladang   
3 -8.10  110.45     Tadah Hujan   
4 -8.10  110.50           Semak   

                                       prediksi_awal  \
0  [{'nama': 'Pisang', 'skor': '85%', 'kategori':...   
1  [{'nama': 'Pisang', 'skor': '89%', 'kategori':...   
2  [{'nama': 'Pisang', 'skor': '83%', 'kategori':...   
3  [{'nama': 'Kacang Hijau', 'skor': '94%', 'kate...   
4  [{'nama': 'Kacang Hijau', 'skor': '78%', 'kate...   

                                   rekomendasi_final  
0  [{'nama': 'Pisang', '